In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. Імпорт бібліотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

Імпортуємо бібліотеки для обробки даних (pandas, numpy), візуалізації (matplotlib, seaborn) та машинного навчання (scikit-learn). Ці бібліотеки є стандартними для задач класифікації на Kaggle.

Висновки:

Використання стандартних бібліотек забезпечує зручну роботу з даними та моделями.
Додавання GridSearchCV і SelectKBest дозволяє оптимізувати модель і зменшити розмірність даних, що може покращити продуктивність порівняно з базовим KNN.

# 2. Завантаження даних

In [ ]:
# Завантаження даних
try:
    train_data = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/train.csv', header=None)
    train_labels = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/trainLabels.csv', header=None)
    test_data = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/test.csv', header=None)
except FileNotFoundError:
    print("Помилка: Перевірте, чи файли доступні за вказаними шляхами.")
    raise

# Перейменування стовпців для зручності
train_data.columns = [f'feature_{i}' for i in range(train_data.shape[1])]
train_labels.columns = ['target']
test_data.columns = [f'feature_{i}' for i in range(test_data.shape[1])]

Завантажуємо три файли: train.csv (40 ознак без назв), trainLabels.csv (бінарні мітки) і test.csv (тестові дані). Оскільки файли не мають заголовків, використовуємо header=None. Перейменовуємо стовпці для зручності роботи (наприклад, feature_0, feature_1, target).

Висновки:

Правильне завантаження даних є критичним, оскільки помилки в шляхах (як попередній FileNotFoundError) зупиняють виконання коду.
Перейменування стовпців полегшує аналіз і робить код більш читабельним.
Використання try-except допомагає діагностувати проблеми з файлами

# 3. Експлораторний аналіз даних (EDA

In [ ]:
# Перегляд перших 5 рядків даних
print("Перші 5 рядків тренувальних даних:")
print(train_data.head())

# Перевірка пропущених значень
print("\nКількість пропущених значень у тренувальних даних:")
print(train_data.isnull().sum())

# Розподіл цільової змінної
print("\nРозподіл цільової змінної:")
print(train_labels['target'].value_counts())

# Гістограми розподілу ознак
train_data.hist(figsize=(15, 15))
plt.show()

# Кореляційна матриця
sns.heatmap(train_data.corr(), cmap='coolwarm')
plt.show()

Проводимо базовий аналіз даних:

Переглядаємо перші рядки для розуміння структури.
Перевіряємо пропущені значення, щоб переконатися, що дані чисті.
Аналізуємо розподіл цільової змінної, щоб виявити можливий дисбаланс класів.
Будуємо гістограми для оцінки розподілу ознак і кореляційну матрицю для виявлення взаємозв’язків між ознаками.
Висновки:

EDA допомагає зрозуміти структуру даних і виявити потенційні проблеми, такі як дисбаланс класів або висока кореляція між ознаками.
У змаганні "Data Science London + Scikit-learn" дані зазвичай чисті, але перевірка пропущених значень є хорошою практикою.
Візуалізації, як-от гістограми та кореляційна матриця, дають змогу оцінити, чи потрібна додаткова обробка даних.

# 4. Попередня обробка даних

In [ ]:
# Попередня обробка даних
X = train_data
y = train_labels['target']

# Масштабування ознак
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
test_scaled = scaler.transform(test_data)

# Відбір ознак
selector = SelectKBest(f_classif, k=20)  # Вибираємо 20 найкращих ознак
X_selected = selector.fit_transform(X_scaled, y)
test_selected = selector.transform(test_scaled)

Виділяємо ознаки (X) і цільову змінну (y).
Масштабуємо ознаки за допомогою StandardScaler, щоб привести їх до однакової шкали, що є важливим для KNN.
Використовуємо SelectKBest з функцією f_classif, щоб вибрати 20 найінформативніших ознак, зменшуючи розмірність і потенційно покращуючи продуктивність моделі.
Висновки:

Масштабування ознак є обов’язковим для KNN, оскільки алгоритм залежить від відстаней між точками.
Відбір ознак зменшує шум і обчислювальну складність, що може підвищити точність порівняно з використанням усіх 40 ознак.
Вибір k=20 є компромісом; експерименти з різними значеннями k можуть дати кращі результати.

# 5. Розділення даних

In [ ]:
# Розділення даних на тренувальний і валідаційний набори
X_train, X_val, y_train, y_val = train_test_split(X_selected, y, test_size=0.2, random_state=42)

Ділимо відібрані ознаки (X_selected) і мітки (y) на тренувальний (80%) і валідаційний (20%) набори. Фіксоване значення random_state=42 забезпечує відтворюваність.

Висновки:

Розділення даних дозволяє оцінити продуктивність моделі на валідаційному наборі перед прогнозуванням на тестових даних.
Співвідношення 80/20 є стандартним, але можна експериментувати з іншими пропорціями (наприклад, 70/30) залежно від розміру даних.

# 6. Навчання моделі KNN

In [ ]:
# Гіперпараметричний пошук для KNN
param_grid = {
    'n_neighbors': range(1, 30),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}
knn = KNeighborsClassifier()
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

# Найкраща модель
best_knn = grid_search.best_estimator_
print("Найкращі параметри:", grid_search.best_params_)

Використовуємо GridSearchCV для підбору оптимальних параметрів KNN: кількість сусідів (n_neighbors), ваги (uniform або distance) і метрику відстані (euclidean або manhattan).
Виконуємо 5-фолдову крос-валідацію для оцінки кожної комбінації параметрів.
Вибираємо модель із найкращими параметрами.
Висновки:

Гіперпараметричний пошук значно покращує продуктивність порівняно з ручним вибором k, як у базовому KNN.
Крос-валідація забезпечує надійнішу оцінку моделі, зменшуючи ризик перенавчання.

# 7. Оцінка моделі

In [ ]:
# Оцінка на валідаційному наборі
y_pred_val = best_knn.predict(X_val)
print("Точність на валідаційному наборі:", accuracy_score(y_val, y_pred_val))
print("\nЗвіт про класифікацію:\n", classification_report(y_val, y_pred_val))
print("\nМатриця плутанини:\n", confusion_matrix(y_val, y_pred_val))

Precision (точність): Частка правильно класифікованих прикладів певного класу серед усіх, які модель віднесла до цього класу.
Для класу 0: 0.95 (95% прикладів, передбачених як клас 0, дійсно належать до нього).
Для класу 1: 0.83 (83% передбачень для класу 1 правильні).
Recall (повнота): Частка правильно класифікованих прикладів певного класу серед усіх прикладів цього класу.
Для класу 0: 0.83 (83% прикладів класу 0 правильно передбачено).
Для класу 1: 0.95 (95% прикладів класу 1 правильно передбачено).
F1-score: Гармонійне середнє між precision і recall.
Для класу 0: 0.88.
Для класу 1: 0.89.
Support: Кількість прикладів у валідаційному наборі для кожного класу (106 для класу 0, 94 для класу 1).
Macro avg: Некваговий середній показник для всіх класів (0.89 для precision, recall, F1).
Weighted avg: Зважений за кількістю прикладів середній показник (0.89 для всіх метрик).
Висновки:

Модель краще передбачає клас 1 (вищий recall = 0.95), але має нижчу precision (0.83), що означає більше помилкових позитивних передбачень для класу 1.
Для класу 0 модель має високу precision (0.95), але нижчий recall (0.83), тобто вона пропускає деякі приклади класу 0.
Класи майже збалансовані (106 проти 94), тому точність є досить репрезентативною метрикою.
F1-score близький до 0.89 для обох класів, що вказує на гарний баланс між precision і recall.

# 8. Прогнозування та створення файлу для відправки

In [ ]:
# Прогнозування на тестовому наборі
y_pred_test = best_knn.predict(test_selected)

# Створення файлу для відправки
submission = pd.DataFrame({
    'Id': range(1, len(y_pred_test) + 1),
    'Solution': y_pred_test
})
submission.to_csv('submission.csv', index=False)
print("Файл submission.csv створено.")

# Загальний висновок
**Продуктивність моделі**
* Точність 88.5%: Я досяг високої точності на валідаційному наборі з 200 прикладів, що означає 177 правильних передбачень із 200. Це вказує на те, що модель ефективно класифікує більшість даних.
* Звіт про класифікацію:
* * Для класу 0: precision = 0.95, recall = 0.83, F1-score = 0.88 (106 прикладів).
  * Для класу 1: precision = 0.83, recall = 0.95, F1-score = 0.89 (94 приклади).
  * Macro avg і weighted avg для precision, recall і F1-score становлять ~0.89, що підтверджує збалансовану продуктивність для обох класів.
**Сильні сторони**
* Збалансована продуктивність: Високий recall для класу 1 (0.95) і висока precision для класу 0 (0.95) показують, що модель ефективно ідентифікує більшість прикладів, особливо для класу 1.
* Обробка даних: Я використав масштабування ознак за допомогою StandardScaler і, ймовірно, відбір ознак (SelectKBest), що забезпечило належну підготовку даних. Це критично важливо для KNN, оскільки алгоритм залежить від відстаней між точками.
* Правильна структура даних: Я коректно завантажив файли train.csv, trainLabels.csv і test.csv за наданими шляхами (/kaggle/input/data-science-london-scikit-learn/), що відповідає умовам змагання.
* Формат відправки: Мій код генерує файл submission.csv із колонками Id і Solution, що відповідає вимогам Kaggle для відправки результатів.
**Висновок** Моя робота з даними та адаптація до умов змагання забезпечила стабільну основу для моделі.
